# Step 3 - metric 점군 BEV 투영 및 도로 차폐 계측

파노라마 1장당 metric 3D 점군을 만들어 360° 가시 프로파일 `r(θ)`를 구하고,
**카메라에서 도로를 따라 40m 이내 구간 중 볼 수 없는 길이의 비율**을 계측한다.

**입력** `output/01_seg`(클래스별 마스크) · `output/02_depth_f16`(disparity) ·
`GIS/road_graph.npz` · `GIS/gangneung_roadsurface.gpkg`(→ `bev/road_prep.py`) ·
`GIS/historical_panoids_filtered.csv`(카메라 위치·방위각·리그 높이)

**출력** `output/03_bev2/{pano}_bev360.jpg` + `{pano}_dsi.json`

## 파이프라인

1. **공동 보정** — 4면 affine 8개 + 공유 지면평면을 동시 적합 (리그 높이는 GIS 에서 고정)
2. **높이대역 점군** — 적합 평면 위 높이 `[1.0, 3.0]m` 인 점만 시선 차단물로 채택
3. **방사 프로파일** — 극좌표 히스토그램 + 누적 투표로 `r(θ)`
4. **pose 정합** — 도로면 BEV 를 실폭도로 폴리곤에 등록해 카메라 위치·방위 보정

## 읽을 때 오해하기 쉬운 두 가지

- **DSI 는 도로 차폐율만으로 만든다.** `D_stop/L_vis` 곱은 뺐다 — 그 항이 DSI 분산의 87%를
  차지해 "인지 불가 구간의 위험도"라는 의도와 반대로 작동했고, 도로 차폐율과 같은 것을 재서
  곱하면 중복 계산이 된다.
- **`sight_flag` 는 `True`/`False`/`null` 세 가지다.** 차량이 구조물보다 가까워 전방을 가리면
  구조적 `L_vis` 는 관측된 값이 아니라 가려진 값이므로 `null`(판정 불가)이 된다.
  집계할 때 `null` 을 `False` 로 취급하면 안 된다.

> 수식·근거·실측치는 `bev/bev_core.py` docstring 과 `bev_redesign_plan.md` 에 있다.

## 0. 패키지 설치

In [1]:
# !pip install opencv-python-headless numpy matplotlib

## 1. 라이브러리 및 BEV 파라미터

In [2]:
import json
import math
import os
import random
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from pyproj import Transformer

# 헬퍼는 bev/ 패키지에 있다. 렌더 워커를 Windows(spawn) 자식 프로세스가 참조로 import 하므로
# 저장소 루트가 sys.path 에 있어야 한다 (노트북을 루트에서 실행하면 보통 이미 들어 있다).
_ROOT = str(Path.cwd())
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)
# **모듈을 강제로 다시 읽는다.** `import` 는 sys.modules 에 이미 있으면 무동작이다.
import importlib
import bev.bev_core
importlib.reload(bev.bev_core)
from bev import bev_core as B      # 보정·투영·프로파일 수식 (독립 스크립트로 검증된 구현을 공유)

FRONT_DIR = Path("output/00_front")
SEG_DIR   = Path("output/01_seg")
DEPTH_DIR = Path("output/02_depth_f16")    # disp_raw(float16). 구 output/02_depth 도 자동 인식
OUT_DIR   = Path("output/260820")   # D-25 판 (리그 높이 상수 2.5). 기존 폴더를 가리키면 RESUME 이 전부 건너뛴다
OUT_DIR.mkdir(parents=True, exist_ok=True)

PANO_STEMS = sorted({p.name[: -len("_front.jpg")] for p in FRONT_DIR.glob("*_front.jpg")})
print(f"파노라마 {len(PANO_STEMS)}개")

TEST_MODE = False   # True: 무작위 20개만 처리 / False: 전체
TEST_N, TEST_SEED = 20, 41
if TEST_MODE:
    PANO_STEMS = sorted(random.Random(TEST_SEED).sample(PANO_STEMS, min(TEST_N, len(PANO_STEMS))))
    print(f"[TEST_MODE] {len(PANO_STEMS)}개만 처리")
RESUME = True and not TEST_MODE

# ── 병렬화 ─────────────────────────────────────────────────────────
# 렌더링(229ms)이 최대 비용이고 CPU matplotlib 작업이라 프로세스 풀로 넘긴다.
# npz 로딩(93ms)은 디스크 I/O + zlib 해제라 GIL 을 놓으므로 스레드로 충분하다(→ 25ms).
RENDER_WORKERS = 12
RENDER_PENDING_MAX = RENDER_WORKERS * 3   # 백프레셔: 미완료 렌더가 무한정 쌓이는 것 방지
LOAD_THREADS = 4                          # 4면 동시 로딩
_loader = ThreadPoolExecutor(max_workers=LOAD_THREADS)

# ── GIS: 도로망 + 실폭도로 폴리곤 (bev/road_prep.py 산출물) ─────────
net = B.RoadNet()
print(f"도로망 정점 {len(net.xy)}개 | 실폭도로 폴리곤 {len(net.surf)}개")

# ── 카메라 pose: 위치(EPSG:5179) + front 면 방위각 + 리그 높이 ─────
# camera_angle[1] 이 front 면의 방위각인지는 실증했다: 실폭도로 폴리곤과의 겹침이
# 오프셋 0°/180° 에서 0.544, 90°/270° 에서 0.40~0.43 -> 90° 어긋남 없음.
# (0/180 모호성은 도메인이 양방향으로 뻗고 L_vis 도 (전방+후방)/2 라 무관하다.)
_cam = pd.read_csv("GIS/historical_panoids_filtered.csv", encoding="utf-8-sig")
_cam["stem"] = "point_" + _cam.point_id.astype(str) + "_pano_" + _cam.pano_id.astype(str)
_cam["bearing"] = _cam.camera_angle.str.strip("[]").str.split(",").str[1].astype(float)
_tf = Transformer.from_crs(4326, B.EPSG, always_xy=True)
_cam["X"], _cam["Y"] = _tf.transform(_cam.pano_lon.values, _cam.pano_lat.values)
CAM = _cam.set_index("stem")[["X", "Y", "bearing"]].to_dict("index")
CAM_H = B.load_cam_heights()
assert {round(v, 2) for v in CAM_H.values()} == {B.CAM_HEIGHT_DEFAULT},     f"리그 높이가 상수가 아니다 -> 스테일 모듈이다 (D-25). 커널을 재시작할 것"
print(f"카메라 pose {len(CAM)}건 | 리그 높이 {sorted({round(v, 2) for v in CAM_H.values()})}")

# 제한속도는 전역 상수가 아니라 도로 등급별로 결정된다 (3절 SPEED_BY_CLASS: 30/50/60 km/h).
V_HEAVY = 0.15      # PoC 기본값 (제안서 4단계 중차량 비율 미정)

print(f"\nR={B.R_MAX}m | 높이대역 {B.H_BAND} | ego {B.R_MIN_EGO}m | {B.N_BINS}방향 | "
      f"투표 {B.MIN_VOTES} | 캔버스 {B.CANVAS}x{B.CANVAS} ({B.GRID_RES:.4f} m/px)")
print(f"렌더 워커 {RENDER_WORKERS} | 로딩 스레드 {LOAD_THREADS} | CPU {os.cpu_count()}코어")

파노라마 36367개
도로망 정점 98486개 | 실폭도로 폴리곤 1655개
카메라 pose 36657건 | 리그 높이 [2.5]

R=40.0m | 높이대역 (1.0, 3.0) | ego {'left': 1.0, 'front': 3.0, 'right': 1.0, 'back': 3.0}m | 720방향 | 투표 3 | 캔버스 240x240 (0.3333 m/px)
렌더 워커 12 | 로딩 스레드 4 | CPU 24코어


## 2. 파노라마 1장 처리

```
load_faces (스레드 4) -> calibrate_panorama -> occupancy x2 (구조물만 / +차량)
  -> ground_bev -> refine_pose -> reachable_surface -> road_occlusion
```

`compute_dsi` 와 `grade_of` 는 아래 3절에서 정의된다. 호출은 4절 실행 루프에서만 일어나므로
셀 순서는 문제되지 않는다.

In [3]:
def load_faces(pano):
    """4면을 스레드로 동시 로딩. npz 해제는 GIL 을 놓으므로 스레드로 충분하다."""
    futs = {d: _loader.submit(B.load_face, pano, d, SEG_DIR, DEPTH_DIR) for d in B.ORDER}
    faces = {d: f.result() for d, f in futs.items()}
    return None if any(v is None for v in faces.values()) else faces


def process_panorama(pano):
    """파노라마 1장 -> (계측 record, 렌더 재료). 입력 부족·보정 실패 시 None."""
    it = CAM.get(pano)
    if it is None:
        return None
    faces = load_faces(pano)
    if faces is None:
        return None

    cam = (it["X"], it["Y"])
    h = CAM_H.get(pano.split("_pano_")[1], B.CAM_HEIGHT_DEFAULT)
    cal = B.calibrate_panorama(faces, h)
    if cal is None:
        return None

    # 구조물만 = 장소의 속성(지도 등급), 차량 포함 = 촬영 순간의 실측
    r_occ, qc = B.occupancy(faces, cal, include_vehicles=False)
    r_veh, _ = B.occupancy(faces, cal, include_vehicles=True)

    # pose 정합: 도로면 BEV 를 실폭도로 폴리곤에 등록.
    # 그림에는 40m 까지 그리고 정합에는 20m 이내만 쓴다 - 평면 투영은 부각이 작아질수록 거리가
    # 폭발하기 때문이다(40m 에서 픽셀당 5.0m = 12.5%). hypot(x,z) >= |z| 이므로 40m 로 뽑아
    # 20m 로 자른 집합은 r_max=20 호출과 정확히 같다 -> 정합 결과 불변.
    mask, half, n = net.surface_mask(cam)
    gx, gy = B.ground_bev(faces, cal, r_max=B.R_MAX, key="road")
    near = np.hypot(gx, gy) <= 20.0
    fit_score, dyaw, dx, dy = B.refine_pose(mask, half, n, gx[near], gy[near], it["bearing"])
    bearing = it["bearing"] + dyaw

    # 도메인 = 도로면을 따라 걸어서 40m 이내인 폴리곤 면적 (D-06). 중심선 위에서는 직선
    # 도로가 자명하게 다 보여 차폐율 0 이 51.8% 로 쏠렸다 -> 면적으로 6.7% (X-38).
    # 정합이 만든 mask 를 그대로 재사용하므로 폴리곤을 다시 굽지 않는다.
    dom, reach = net.reachable_surface(mask, half, cam, dx, dy)
    # r_veh 를 함께 넘겨 '차량 뒤라 알 수 없는' 도로를 분모에서 뺀다. 구조물만으로 재면 차량이
    # 가린 방향은 그 뒤가 안 보여 r_occ 가 커지고 '도로가 보인다'로 계수된다 (평균 6.6%p 낙관).
    ro = B.road_occlusion(dom, r_occ, cam, bearing, dx, dy, r_veh=r_veh)
    ro_v = B.road_occlusion(dom, r_veh, cam, bearing, dx, dy)
    if ro is None or ro_v is None:
        return None

    # L_vis 는 전/후방의 **작은 쪽**이다 (D-20). 셔틀은 링크를 양방향으로 지날 수 있고,
    # 한 방향이라도 D_stop 을 못 채우면 그 방향 주행에서는 멈출 수 없다.
    f, b, lv = B.l_vis(r_occ)
    f_v, b_v, lv_v = B.l_vis(r_veh)

    # 연속 위험도는 도로 차폐율만으로, 가시거리는 지시함수로 분리 (윗 셀 주석 참조)
    dsi = compute_dsi(ro["road_occluded_frac"], V_HEAVY)
    dsi_v = compute_dsi(ro_v["road_occluded_frac"], V_HEAVY)

    # 지면이 보이는 면이 2개 미만이면 보정이 식별되지 않는다 -> 값을 내지 않는다.
    # 특히 0면은 점이 거의 R_MAX 를 통과하지 못해 모든 방향을 '트임'으로 보고하고,
    # 결과적으로 최하위 등급(=안전)으로 찍힌다. 보정 실패를 안전이라 말하는 셈이다.
    # dsi/grade 를 None 으로 두면 build_dsi_map.py 가 그대로 건너뛴다(웹 변경 불필요).
    # 이음매 4쌍을 개별 보존한다. 중앙값만 남기면 최악 이음매가 가려진다
    # (실측: 중앙 <= 0.055 인데 최악 > 0.10 인 경우가 23.4%). 계산은 어차피 4번 한다.
    seam = B.seam_residuals(faces, cal)
    seam_ok = [v for v in seam.values() if v is not None]

    faces300 = B.faces_with_ground(cal)
    calib_ok = B.calib_valid(cal)
    road_ok = ro["road_unknown_frac"] <= B.ROAD_UNKNOWN_MAX   # 절반 넘게 모르면 판정 안 함
    pose_clip = B.pose_clipped(dyaw, dx, dy)                  # 기록 전용 (D-16 폐기, X-25)
    # 기록 전용: 원거리(20~40m)와 전/후방을 쪼갠 포함률. fit_score 하나로는 원거리 어긋남과
    # 회전 오차가 상쇄되어 안 보인다. 폭 교란이 남아 있어 판정에는 쓰지 않는다.
    fit_bd = B.fit_breakdown(mask, half, n, gx, gy, it["bearing"], dyaw, dx, dy)
    # 이 지점이 우리가 가진 도로 지도 위에 있는가. 밖이면 GIS 도로를 어디에 놓아도 근거가 없다.
    # 옛 기준(pose 가 탐색 한계에 붙음)은 within-edge 판별력이 0 이었다 -> D-19/X-25.
    d_surf = net.dist_to_surface(cam)
    on_road = d_surf <= B.ON_ROAD_MAX
    valid = calib_ok and road_ok and on_road
    # 하드 게이트를 통과한 뒤의 품질 등급. 판정 여부와 무관하다 -- 세 축 모두 위험을
    # **높게** 말하는 방향이라 버리지 않고 등급만 남긴다. 노선 최적화가 가중치를 낮춘다 (D-24).
    seam_mx = float(max(seam_ok)) if seam_ok else None
    conf = B.confidence(d_surf, seam_mx if seam_mx is not None else 0.0,
                        fit_bd["fit_fwd"], fit_bd["fit_bwd"])

    cls = ROAD_CLS.get(pano)
    speed = B.SPEED_BY_CLASS.get(cls, B.SPEED_DEFAULT)
    d_stop = B.stopping_distance(speed)

    # 차량 뒤가 무엇인지는 모르지만 그 불확실성에는 경계가 있다.
    # r_veh <= r_occ 가 항상 성립하므로(차단물을 더하면 가시거리는 짧아질 뿐, 실측 위반 0건)
    # 진짜 '구조적' 가시거리는 [lv_v, lv] 구간 안에 있다:
    #   하한 lv_v -- 차량 뒤의 벽이라도 최소한 차량만큼은 멀다
    #   상한 lv   -- 차량이 없었다면 구조물까지 다 보였을 것이다
    # 구간이 D_stop 한쪽에 통째로 놓이면 차량 뒤를 몰라도 결론이 바뀌지 않는다.
    # 걸칠 때만 판정을 보류한다. 이 규칙으로 판정 불가가 7.2% -> 1.2% 로 준다. (D-17)
    vb_f, vb_b = B.vehicle_blocked_frac(r_occ, r_veh)   # 기록용 (판정 기준은 아니다)
    if lv < d_stop:
        sight_flag = True                      # 상한조차 미달 -> 차량과 무관하게 미확보
    elif lv_v > d_stop:
        sight_flag = False                     # 하한조차 초과 -> 차량이 있어도 확보
    else:
        sight_flag = None                      # 구간이 D_stop 을 걸친다

    rec = {
        "pano": pano,
        "dsi_refined": round(dsi, 4) if valid else None,               # 웹 계약면
        "grade": grade_of(dsi) if valid else None,
        "dsi_veh": round(dsi_v, 4) if valid else None,
        "grade_veh": grade_of(dsi_v) if valid else None,
        "valid": valid, "calib_valid": calib_ok, "road_known_enough": road_ok,
        "on_mapped_road": on_road, "cam_off_road_m": round(d_surf, 2),
        "confidence": conf,                                            # 2 높음 / 1 보통 / 0 낮음 (D-24)
        "dsi_raw": round(dsi, 4),                                      # 진단용 (판정 불가여도 기록)
        # ── 연속 위험도의 근거 ──
        "road_occluded_frac": round(ro["road_occluded_frac"], 4),
        "road_occluded_m2": round(ro["road_occluded_m2"], 1),
        "road_domain_m2": round(ro["road_domain_m2"], 1),
        "road_known_m2": round(ro["road_known_m2"], 1),
        "road_unknown_frac": round(ro["road_unknown_frac"], 4),   # 차량 뒤라 판정에서 뺀 비율
        "road_occluded_frac_veh": round(ro_v["road_occluded_frac"], 4),
        # ── 정지시거 지시함수 (원래 의도의 레드 플래그) ──
        # sight_flag 는 구조물 기준 = 장소의 속성. 촬영 당일 교통에 등급이 좌우되면 안 된다.
        # sight_flag_veh 는 그 순간의 실측 (참고용).
        "sight_flag": sight_flag,
        "sight_flag_veh": bool(lv_v < d_stop),
        "veh_blocked_fwd": round(vb_f, 3), "veh_blocked_bwd": round(vb_b, 3),
        "l_vis_margin_m": round(lv - d_stop, 2),      # 음수면 미확보. 경계 사례 식별용
        "road_class": cls, "speed_limit_kmh": speed, "d_stopping_m": round(d_stop, 2),
        "l_vis_m": round(lv, 2), "l_vis_fwd_m": round(f, 2), "l_vis_bwd_m": round(b, 2),
        "l_vis_veh_m": round(lv_v, 2),
        "l_vis_censored": bool(lv >= B.R_MAX - 1e-6),
        # ── 참고: 원판 도메인 (도메인을 되돌려 비교할 때) ──
        "a_shadow_disk_m2": round(B.shadow_area(r_occ), 1),
        "a_total_disk_m2": round(math.pi * B.R_MAX ** 2, 1),
        "r_max_m": B.R_MAX, "h_band_m": list(B.H_BAND), "v_heavy": V_HEAVY,
        "cam_height_m": round(h, 2),
        "calib": {"a": [round(v, 6) for v in cal["a"]], "b": [round(v, 6) for v in cal["b"]],
                  "pitch_deg": round(cal["pitch_deg"], 3), "roll_deg": round(cal["roll_deg"], 3),
                  "seam_rms_rel": round(float(np.median(seam_ok)), 4) if seam_ok else None,
                  "seam_max": round(float(max(seam_ok)), 4) if seam_ok else None,
                  "seam_worst": (max(((v, k) for k, v in seam.items() if v is not None))[1]
                                 if seam_ok else None),
                  "seam_per": {k: (round(v, 4) if v is not None else None)
                               for k, v in seam.items()},
                  "n_ground_px": cal["n_ground"], "faces300": faces300,
                  "status": cal["status"]},
        "pose": {"fit_score": round(fit_score, 4), "dyaw_deg": dyaw, "dx_m": dx, "dy_m": dy,
                 "clipped": pose_clip, **fit_bd},
        "coverage": {"n_points": qc["n_points"], "n_raw": qc["n_raw"],
                     "n_mask": qc["n_mask"],
                     "starved_bins": qc["starved_bins"], "empty_bins": qc["empty_bins"]},
        "method_version": "03-bev-v2",
    }

    # 렌더 재료. GIS 도로면(폴리곤 실제 모양) 위에 사진이 본 도로면을 겹쳐 그린다 - 지표가
    # 의존하는 GIS 배치가 이 파노라마에서 맞았는지를 그림만 보고 판정할 수 있게 하기 위해서다.
    #   회색만 있고 파랑 없음   -> 가려서 못 본 도로 (측정하려는 것)
    #   파랑이 회색 밖으로 나감 -> 정합 오차 (믿으면 안 되는 것)
    # road_mask 를 넘기면 워커가 사각지대를 도로 위(계수됨)/도로 밖(맥락)으로 나눠 칠한다.
    road_mask = B.surface_grid(mask, half, cam, bearing, dx, dy)
    bev = B.bev_canvas(faces, cal, mask, half, reach, (gx, gy), cam, bearing, dx, dy)

    render = dict(bev=bev, road_mask=road_mask,
                  shadow_occ=B.shadow_grid(r_occ), shadow_veh=B.shadow_grid(r_veh),
                  ray_hits_occ=B.ray_hits(r_occ), ray_hits_veh=B.ray_hits(r_veh))
    return rec, render, (r_occ, r_veh)


print("파노라마 처리 함수 정의 완료")

파노라마 처리 함수 정의 완료


## 3. 지표 정의

In [4]:
def compute_dsi(road_occluded_frac, v_heavy=0.0):
    """연속 위험도. 순위·tercile 용. 범위 [1, 2] x (1+v_heavy)."""
    return (1.0 + road_occluded_frac) * (1.0 + v_heavy)


def grade_of(dsi):
    """웹은 이 값을 무시하고 자체 tercile 로 등급을 매긴다(PointDetailPanel.tsx).
    build_dsi_map.py 가 non-null 을 요구해 필드만 유지한다. 실행 후 tercile 재적합이 정식 절차."""
    return "Safe" if dsi < 1.0 else ("Caution" if dsi < 1.8 else "High-risk")


# 도로 등급 -> 제한속도 -> D_stop. 근거는 bev_core.SPEED_BY_CLASS 주석 참조
# (도로명주소법 시행령 제6조 대로/로/길 + 안전속도 5030).
ROAD_CLS = B.load_road_class()
print("도로 등급 분포:", pd.Series(list(ROAD_CLS.values())).value_counts().to_dict())
for c, label in [("4", "길(이면도로)"), ("3", "로(일반도로)"), ("2", "대로")]:
    s = B.SPEED_BY_CLASS[c]
    d = B.stopping_distance(s)
    print(f"  '{c}' {label:<12} {s:>4.0f} km/h -> D_stop {d:5.1f} m"
          f"  ({'OK' if d < B.R_MAX else 'R_MAX 초과 - 판정 불가'})")

도로 등급 분포: {'4': 24647, '3': 10520, '2': 1490}
  '4' 길(이면도로)        30 km/h -> D_stop  13.4 m  (OK)
  '3' 로(일반도로)        50 km/h -> D_stop  27.9 m  (OK)
  '2' 대로             60 km/h -> D_stop  36.9 m  (OK)


## 4. 전체 실행

계산은 메인 프로세스에서 순차 처리하고, 결과 배열만 워커 프로세스로 넘겨 렌더링·저장을 맡긴다.
`bev/bev_render_worker.py` 는 수정하지 않았다 — 출력 JPEG 형상(649×2187)이 유지되어야
웹의 CSS 크롭이 깨지지 않는다.

> **실행 후 웹 반영 순서**: ① 새 분포로 tercile 재적합 → ② `PointDetailPanel.tsx` 의
> `POINT_DSI_TERCILES` 와 `MapView.tsx` 의 `ROAD_DSI_TERCILES` 갱신 → ③ `build_dsi_map.py` 의
> `BEV_DIR` 를 `output/03_bev2` 로 변경. **재적합 전에 ③을 하면 이동한 분포를 옛 임계로
> 등급화하게 된다.** `build_road_dsi_map.py` 도 옛 임계(1.0/1.8)를 하드코딩하고 있어 같이 고쳐야 한다.

In [5]:
import warnings
from concurrent.futures import ProcessPoolExecutor

from tqdm.auto import tqdm

# 수정 없음: r(theta) 가 워커 규약([(각도,거리)])에 그대로 맞는다.
# Windows(spawn) 자식은 이 함수를 참조로 import 하므로 bev/ 가 sys.path 에서 보여야 한다.
from bev.bev_render_worker import render_and_save

warnings.filterwarnings("ignore")

N_ARROWS = 12   # 범례는 B.BEV_LEGEND (캔버스 색 규약과 한 곳에서 관리)

dsi_summary = []
n_skipped = 0

# 이어서 실행 지원: 별도 진행 기록 없이 출력 폴더 상태로만 재개 지점을 판단한다.
# 이진 탐색을 쓰지 않는 이유: 입력이 부족하거나 도로망이 ±R_MAX 안에 없는 파노라마는
# 아무 파일도 남기지 않고 건너뛰므로(약 1%), 완료 항목이 앞쪽 연속 구간이라는 단조성이
# 깨진다. 구멍에서 탐색이 멈춰 이미 끝난 뒷부분까지 통째로 재처리하게 된다.
# 두 파일이 모두 있는 것만 완료로 보고 집합 차집합을 쓴다(글롭 2회, 약 1초).
remaining = PANO_STEMS
if RESUME:
    done = ({p.name[: -len("_dsi.json")] for p in OUT_DIR.glob("*_dsi.json")}
            & {p.name[: -len("_bev360.jpg")] for p in OUT_DIR.glob("*_bev360.jpg")}
            & {p.name[: -len("_rtheta.npz")] for p in OUT_DIR.glob("*_rtheta.npz")})
    remaining = [p for p in PANO_STEMS if p not in done]
    print(f"이미 완료: {len(PANO_STEMS) - len(remaining)}개 / 전체: {len(PANO_STEMS)}개 "
          f"→ 남은 작업: {len(remaining)}개")
else:
    print(f"전체 {len(PANO_STEMS)}개 처리 (RESUME 꺼짐)")

executor = ProcessPoolExecutor(max_workers=RENDER_WORKERS)
pending = []


def _drain(n_keep):
    """완료된 렌더 future 를 걷어내고, pending 이 n_keep 이하가 될 때까지 앞에서부터 대기."""
    while len(pending) > n_keep:
        pending[0].result()      # 예외가 있으면 여기서 즉시 표면화
        pending.pop(0)


try:
    for pano in tqdm(remaining, desc="BEV 도로차폐"):
        out = process_panorama(pano)
        if out is None:
            n_skipped += 1
            continue
        rec, render, (r_occ, r_veh) = out
        dsi_summary.append(rec)

        # _dsi.json 은 가볍고, RESUME 일관성을 위해 메인 프로세스에서 즉시 기록
        json.dump(rec, open(OUT_DIR / f"{pano}_dsi.json", "w", encoding="utf-8"),
                  ensure_ascii=False, indent=2)
        # r(theta) 를 남기면 도메인 실험(중심선 1-D vs 폴리곤 면적 2-D)을 재실행 없이 할 수
        # 있다 - road_occlusion 은 r_theta + GIS 만 쓰고 depth/seg 가 필요 없다.
        # 720칸 x2 float16 = 2.9KB/장, 전량 약 70MB (JPEG 4.1GB 대비 무시할 수준).
        np.savez_compressed(OUT_DIR / f"{pano}_rtheta.npz",
                            r_occ=r_occ.astype(np.float16), r_veh=r_veh.astype(np.float16))

        # sight_flag 는 True / False / None(차량에 가려 판정 불가) 세 가지다
        sf = rec["sight_flag"]
        tag = "  [SIGHT FLAG]" if sf else ("  [flag n/a: vehicles]" if sf is None else "")
        args = dict(
            **render,
            center=B.CENTER, grid_res=B.GRID_RES, include_vehicles=True, n_arrows=N_ARROWS,
            out_path=str(OUT_DIR / f"{pano}_bev360.jpg"),
            # 제목은 bev_core 가 만든다 (검증한 문자열과 실제 문자열을 일치시키기 위해)
            suptitle=B.suptitle_for(rec, pano, tag),
            title_occupancy=f"1. Occupancy  ({B.GRID_RES:.2f}m/px, r={B.R_MAX:.0f}m)"
                            f"\nlight=GIS road  dark=measured domain  blue=seen in photo",
            # 수치 반복 금지: occluded/DSI 는 suptitle 에만. 여기엔 패널 고유 정보만 둔다.
            title_occ=(f"2. Blind zone - structures only  (used for grading)"
                       f"\nL_vis fwd {rec['l_vis_fwd_m']:.1f}m / bwd {rec['l_vis_bwd_m']:.1f}m"),
            title_veh=(f"3. Blind zone - vehicles included  (capture-day snapshot)"
                       f"\nroad occluded {rec['road_occluded_frac_veh']:.1%}"
                       f"  ->  DSI {compute_dsi(rec['road_occluded_frac_veh'], V_HEAVY):.2f}"),
            legend_colors=B.BEV_LEGEND,
        )
        pending.append(executor.submit(render_and_save, args))
        _drain(RENDER_PENDING_MAX)     # 백프레셔

        del out, render, args

    _drain(0)                          # 남은 렌더 전부 완료 대기
finally:
    executor.shutdown(wait=True)

print(f"\n=== 완료: {len(dsi_summary)}개 처리, 건너뜀 {n_skipped}개 ===")
if dsi_summary:
    df = pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, dict)} for r in dsi_summary])
    df["seam_max_"] = [r["calib"]["seam_max"] or 0.0 for r in dsi_summary]
    df["near_"] = [np.nanmean([v for v in (r["pose"]["fit_fwd"], r["pose"]["fit_bwd"])
                               if v is not None] or [np.nan]) for r in dsi_summary]
    q = lambda s: f"p10 {s.quantile(.1):.3f} | p50 {s.median():.3f} | p90 {s.quantile(.9):.3f}"
    print(f"판정 불가 {(~df.valid).mean():.1%}"
          f"  = 보정(faces300<2) {(~df.calib_valid).mean():.1%}"
          f" | 도로 미상 {(~df.road_known_enough).mean():.1%}"
          f" | 도로지도 밖 {(~df.on_mapped_road).mean():.1%}   (사유 중복 포함)")
    print(f"차량 뒤 도로 비율 : {q(df.road_unknown_frac)}")
    print(f"도로 차폐율 : {q(df.road_occluded_frac)}  (분모 = 아는 도로)")
    print(f"DSI         : {q(df.dsi_refined)}")
    _sp = pd.Series([r["calib"]["seam_max"] for r in dsi_summary if r["calib"]["seam_max"] is not None])
    print(f"이음매 최악    : {q(_sp)}   (중앙값 기준 G1 통과와 별개로 볼 것)")
    print(f"L_vis 상한걸림 {df.l_vis_censored.mean():.1%} | 보정 status "
          f"{pd.Series([r['calib']['status'] for r in dsi_summary]).value_counts().to_dict()}")
    print(f"pose fit_score: {q(pd.Series([r['pose']['fit_score'] for r in dsi_summary]))}")
    print(f"카메라 도로지도 이탈: 중앙 {df.cam_off_road_m.median():.2f}m"
          f" | >{B.ON_ROAD_MAX}m {(df.cam_off_road_m > B.ON_ROAD_MAX).mean():.1%}"
          f" | 탐색한계 2개(기록용) {pd.Series([r['pose']['clipped'] for r in dsi_summary]).ge(2).mean():.1%}")

    # 신뢰도는 게이트가 아니다 - 값은 다 나오고 등급만 붙는다 (D-24).
    # 낮을수록 위험을 높게 말하는 쪽으로 치우친다: 실측 초과차폐 0 +0.020 / 1 +0.005 / 2 -0.006
    _cf = df[df.valid].confidence
    print(f"신뢰도(판정 가능 {len(_cf)}개): "
          + " | ".join(f"{k}={v/len(_cf):.1%}" for k, v in sorted(_cf.value_counts().items(), reverse=True))
          + f"   |  낮음 사유(중복): 도로지도 밖>{B.CONF_OFF_LOW}m "
            f"{(df.valid & (df.cam_off_road_m > B.CONF_OFF_LOW)).sum()/len(_cf):.1%}"
            f" / 이음매>={B.CONF_SEAM_LOW} "
            f"{(df.valid & (df.seam_max_ >= B.CONF_SEAM_LOW)).sum()/len(_cf):.1%}"
            f" / near<{B.CONF_NEAR_LOW} "
            f"{(df.valid & (df.near_ < B.CONF_NEAR_LOW)).sum()/len(_cf):.1%}")

    # sight_flag: True/False/None(차량에 가려 판정 불가). None 은 평균에서 빼야 한다.
    sf = df.sight_flag
    n_na = sf.isna().sum()
    valid = sf.dropna().astype(bool)
    print(f"\n정지시거 지시함수: 판정 가능 {len(valid)}개 중 플래그 {valid.mean():.1%}"
          f" | 판정 불가(차량 차폐) {n_na / len(df):.1%}")
    for c, label in [("4", "길(30km/h)"), ("3", "로(50km/h)"), ("2", "대로(60km/h)")]:
        s = df[df.road_class == c]
        v = s.sight_flag.dropna().astype(bool)
        if len(s):
            print(f"  '{c}' {label:<13} {len(s):>6}개 | 플래그 {v.mean() if len(v) else float('nan'):>6.1%}"
                  f" | 판정불가 {s.sight_flag.isna().mean():>5.1%}"
                  f" | 여유 중앙값 {s.l_vis_margin_m.median():+6.1f}m")
    # 경계 불안정 점검: |여유| 가 작으면 측정오차로 플래그가 뒤집힐 수 있다
    print(f"  임계 ±3m 이내(플래그 불안정 구간): {(df.l_vis_margin_m.abs() < 3.0).mean():.1%}")
    print(f"  참고) 차량 포함 기준 플래그: {df.sight_flag_veh.mean():.1%}"
          f" | 전방 차량차폐 비율 중앙값 {df.veh_blocked_fwd.median():.2f}")

    print("\n다음: ① 새 분포로 등급 임계 재적합(D-11) → ② web/app/versions.ts 갱신"
          " → ③ build_dsi_map.py 의 BEV_DIR 를 output/260819_2 로 변경")

c:\Users\Jihyun\Desktop\git\ITS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


이미 완료: 0개 / 전체: 36367개 → 남은 작업: 36367개


BEV 도로차폐:   0%|          | 0/36367 [00:00<?, ?it/s]


NameError: name 'links' is not defined